# India AlphaEarth LORO — 10-city panel

Week4-parity leave-one-city-out on the expanded India metro set (5 original + 5 new).

**Run:** `india_loro_20261005_202600`  
**n cells (balanced):** 1,071,878 across 10 cities  
**Runtime:** 251s · workers=20  
**Protocol:** pure-exclusive clear cells · per-city 1:1 balance (seed 42) · `StandardScaler` on train only · threshold 0.5 · lr64 (+ PCA/PLS probes)

Figures are pre-rendered under `../figures/india_loro/` — open this notebook without re-running to view results.


## 1. Headline

| Metric | Value |
|---|---:|
| India 10-city LORO (mean lr64 F1) | **0.523** |
| Within-India 80/20 split | **0.707** |
| City→city off-diagonal mean | **0.454** |
| Prior 5-city India LORO (intl run) | 0.437 |
| USA LORO (intl run) | 0.860 |

![headline](../figures/india_loro/01_headline.png)

**Read:** Expanding from 5→10 cities lifts mean LORO (~0.44→0.52) but India remains far below USA. The within-country pooled head (0.71) is much stronger than leave-one-city-out — geographic fragmentation, not a weak classifier.


## 2. Per-city LORO

Train on the other 9 Indian cities; evaluate held-out metro.

| Held-out | macro-F1 | AUROC | n |
|---|---:|---:|---:|
| Ahmedabad | 0.512 | 0.651 | 96,090 |
| Bengaluru | 0.455 | 0.737 | 348,210 |
| Chennai | 0.389 | 0.699 | 62,658 |
| Delhi | 0.595 | 0.715 | 157,894 |
| Hyderabad | 0.546 | 0.776 | 72,136 |
| Jaipur | 0.573 | 0.612 | 6,204 |
| Kolkata | 0.369 | 0.712 | 24,824 |
| Lucknow | 0.519 | 0.634 | 7,916 |
| Mumbai | 0.654 | 0.746 | 202,108 |
| Pune | 0.617 | 0.769 | 93,838 |

Hardest: **Kolkata** (0.369). Easiest: **Mumbai** (0.654).

![loro by city](../figures/india_loro/02_loro_by_city.png)


## 3. PCA / PLS probes

Mean macro-F1 over the same 10 holdouts.

| model | mean F1 | std |
|---|---:|---:|
| lr64(k=64) | 0.523 | 0.095 |
| pca(k=2) | 0.424 | 0.071 |
| pca(k=9) | 0.594 | 0.060 |
| pls(k=2) | 0.598 | 0.050 |
| pls(k=9) | 0.551 | 0.063 |

![probes](../figures/india_loro/03_probe_comparison.png)

PLS-2 / PCA-9 beat raw lr64 on mean LORO — low-rank supervised / unsupervised projections help a bit, but do not close the gap to USA (~0.86).


## 4. City → city transfer heatmap

Train on one city, test on each other (diagonal = train/test same city). Off-diagonal mean F1 = **0.454** (n=90 pairs).

![city heat](../figures/india_loro/04_city_transfer_heatmap.png)

**Worst pairs**

| pair | F1 |
|---|---:|
| Ahmedabad → Pune | 0.333 |
| Lucknow → Delhi | 0.333 |
| Kolkata → Chennai | 0.333 |
| Ahmedabad → Hyderabad | 0.333 |
| Ahmedabad → Kolkata | 0.333 |
| Ahmedabad → Mumbai | 0.333 |
| Delhi → Chennai | 0.333 |
| Lucknow → Pune | 0.333 |

**Best pairs**

| pair | F1 |
|---|---:|
| Pune → Mumbai | 0.723 |
| Hyderabad → Bengaluru | 0.703 |
| Mumbai → Pune | 0.682 |
| Mumbai → Bengaluru | 0.656 |
| Kolkata → Delhi | 0.647 |
| Chennai → Mumbai | 0.645 |
| Hyderabad → Pune | 0.642 |
| Hyderabad → Kolkata | 0.628 |


## 5. Sample sizes & F1–AUROC

Balanced extract is 1:1 per city; metros differ ~50× in n (Jaipur/Lucknow small, Bengaluru/Mumbai large).

![sizes](../figures/india_loro/05_sample_sizes.png)

![f1 auroc](../figures/india_loro/06_f1_vs_auroc.png)

Several cities show mid AUROC with low decision F1 (class-conditional score shift / threshold mismatch under a shared Indian head).


## Takeaways

1. **10-city India LORO mean lr64 F1 = 0.523** — better than the 5-city panel (~0.44), still well below USA (~0.86) and below within-India 80/20 (0.707).
2. **Fragmentation, not capacity:** a pooled Indian head works in-distribution; leave-one-city-out and city→city transfer (mean 0.454) show metro-specific decision boundaries.
3. **Hard / easy poles:** Kolkata (0.369) vs Mumbai (0.654).
4. **Probes:** PCA-9 / PLS-2 raise mean LORO into the mid-0.5s–0.6s; still not USA-like sharing.
5. **Next:** city-conditioned heads, AOI/label audits on weak metros, or subspace (global⊕local) before claiming India↔USA transfer.

Artifacts: `/data/users/kv133/Alpha-Earth-Embeddings-Research/week4/international_geo_generalization/runs/india_loro_20261005_202600/artifacts/` · report: `/data/users/kv133/Alpha-Earth-Embeddings-Research/week4/international_geo_generalization/runs/india_loro_20261005_202600/RESULTS.md`


## Reproduce / reload tables

In [ ]:
from pathlib import Path
import json
import pandas as pd

RUN = Path("/data/users/kv133/Alpha-Earth-Embeddings-Research/week4/international_geo_generalization/runs/india_loro_20261005_202600").resolve()
if not RUN.exists():
    # fallback relative to this notebook
    RUN = (Path.cwd() / "../runs/india_loro_20261005_202600").resolve()
ART = RUN / "artifacts"
loro = pd.read_csv(ART / "loro_IND_metrics.csv")
city = pd.read_csv(ART / "city_transfer_metrics.csv")
country = pd.read_csv(ART / "country_transfer_metrics.csv")
counts = pd.read_csv(RUN / "extract" / "city_counts.csv")
summary = json.loads((RUN / "summary.json").read_text())
print(summary)
display(loro[loro.model == "lr64"][["held_out", "macro_f1", "auroc", "n"]].sort_values("macro_f1"))


### Optional: regenerate figures + this notebook

In [ ]:
# !../../.venv/bin/python ../scripts/build_india_loro_notebook.py --run-dir ../runs/india_loro_20261005_202600
